# Complex mask mel stft blend training

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook tests complex-mask blending between mel and STFT routes.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


In [ ]:
# =========================
# Installs / imports
# =========================
import os, sys, json, time, math, random, shutil, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader
from IPython.display import Audio, display, Markdown

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# =========================
# Mount Drive
# =========================
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", repr(e))

In [ ]:
# =========================
# Repro / config
# =========================
SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/master")
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"
assert DRIVE_ROOT.exists(), f"Missing DRIVE_ROOT: {DRIVE_ROOT}"

MODE = "A1CMASK"
RUN_NAME = "complex_mask_mel_stft_blend"
RUN_TAG = f"{RUN_NAME}_{time.strftime('%Y%m%d_%H%M%S')}"
CKPT_DIR = DRIVE_ROOT / "checkpoints_runs" / RUN_TAG
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV = CKPT_DIR / "train_log.csv"

# -------------------------
# Dataset / training
# -------------------------
SR = 22050
SEG_S = 2.0
BATCH = 1                 # safest for BigVGAN + STFT blend; set to 2 if memory is fine
P_SPEECH = 0.70
NUM_WORKERS = 2
PIN_MEMORY = (device.type == "cuda")

STEPS = 25000
SAVE_EVERY = 2500
LOG_EVERY = 100

# -------------------------
# Mel frontend / BigVGAN
# -------------------------
N_MELS = 80
N_FFT = 1024
HOP = 256
WIN = 1024
FMIN = 0
FMAX = 8000
CENTER_MEL = False

BIGVGAN_MODEL_ID = "nvidia/bigvgan_v2_22khz_80band_fmax8k_256x"
BIGVGAN_USE_CUDA_KERNEL = False

# -------------------------
# STFT teacher and blend-grid config
# -------------------------
# Keep these the same unless the STFT teacher used a different setting.
CPLX_N_FFT = 1024
CPLX_HOP = 256
CPLX_WIN = 1024
CPLX_CENTER = True

# The actual complex-mask blend happens on this grid.
# Usually keep it equal to the complex teacher grid.
BLEND_N_FFT = 1024
BLEND_HOP = 256
BLEND_WIN = 1024
BLEND_CENTER = True
BLEND_F_BINS = BLEND_N_FFT // 2 + 1

# -------------------------
# Inpainting
# -------------------------
K_CHOICES = (1, 2)

# -------------------------
# Fusion architecture
# -------------------------
G_BASE = 24
G_GROUPS = 8
FUSION_IN_CH = 7
MASK_TEMP = 1.0
MASK_REAL_MAX = 1.20       # real part is in [0, MASK_REAL_MAX]
MASK_IMAG_SCALE = 0.75     # imaginary part is in [-scale, scale]
INIT_ALPHA_BIAS = -4.0     # starts very close to mel route; less risk of weird early blends

# -------------------------
# Loss weights
# -------------------------
# The main objective is waveform/STFT quality while preserving mel-route clarity.
LAMBDA_LOGSTFT_HF = 25.0
LAMBDA_COMPLEX = 0.25
LAMBDA_PHASE = 1.0
LAMBDA_WAV_MRSTFT = 2.0

LAMBDA_MEL_RECON = 8.0
LAMBDA_MEL_HF = 25.0
LAMBDA_MEL_TDIFF = 8.0

# Small penalties so the mask does not overuse STFT everywhere.
LAMBDA_MASK_REAL = 0.004
LAMBDA_MASK_IMAG = 0.002
LAMBDA_BLEND_DELTA = 0.001

# HF weighting and prior
HF_START_FRAC = 0.45
HF_END_GAIN = 8.0
HF_RAMP_POWER = 2.0

STFT_HF_START_FRAC = 0.45
STFT_HF_END_GAIN = 6.0
STFT_HF_RAMP_POWER = 2.0

MASK_DILATE = 3
TDIFF_MASK_DILATE = 3
PRIOR_RADIUS = 3
PRIOR_BOUNDARY_GAIN = 1.50
PRIOR_HF_POWER = 1.30

# Waveform MRSTFT loss
WAV_MRSTFT_FFTS = (512, 1024, 2048)
WAV_MRSTFT_HOPS = (128, 256, 512)
WAV_MRSTFT_WINS = (512, 1024, 2048)

# -------------------------
# Teacher checkpoints
# -------------------------
# The mel teacher should be the stable mel 2D U-Net refiner checkpoint.
# Leave as None to auto-detect latest run with MEL_TEACHER_PREFIX, or paste a full .pt path.
MEL_TEACHER_CKPT = None
MEL_TEACHER_PREFIX = None  # Example: "teacher_run_prefix"

# Leave as None to auto-detect latest complex STFT teacher, or paste a full .pt path.
COMPLEX_TEACHER_CKPT = None
COMPLEX_TEACHER_PREFIX = None  # Example: "teacher_run_prefix"

# Run one small forward/backward check before the real training loop.
RUN_SMOKE_TEST_BEFORE_TRAINING = True

# Resume this complex-mask model itself
RESUME_PATH = None
AUTO_RESUME_SAME_FAMILY = False
RESUME_LR_SCALE = 0.75

LR_G = 1.5e-4
WEIGHT_DECAY = 1e-4

RUN_CONFIG = dict(
    seed=SEED,
    mode=MODE,
    run_name=RUN_NAME,
    sr=SR,
    segment_s=SEG_S,
    batch=BATCH,
    p_speech=P_SPEECH,
    mel=dict(n_fft=N_FFT, hop=HOP, win=WIN, n_mels=N_MELS, fmin=FMIN, fmax=FMAX, center=CENTER_MEL),
    complex_teacher_stft=dict(n_fft=CPLX_N_FFT, hop=CPLX_HOP, win=CPLX_WIN, center=CPLX_CENTER),
    blend_stft=dict(n_fft=BLEND_N_FFT, hop=BLEND_HOP, win=BLEND_WIN, center=BLEND_CENTER),
    k_choices=list(K_CHOICES),
    arch=dict(
        g_arch="ComplexMaskFusionUNet2D",
        g_base=G_BASE,
        g_groups=G_GROUPS,
        fusion_in_ch=FUSION_IN_CH,
        mask_temp=MASK_TEMP,
        mask_real_max=MASK_REAL_MAX,
        mask_imag_scale=MASK_IMAG_SCALE,
        init_alpha_bias=INIT_ALPHA_BIAS,
    ),
    losses=dict(
        lambda_logstft_hf=LAMBDA_LOGSTFT_HF,
        lambda_complex=LAMBDA_COMPLEX,
        lambda_phase=LAMBDA_PHASE,
        lambda_wav_mrstft=LAMBDA_WAV_MRSTFT,
        lambda_mel_recon=LAMBDA_MEL_RECON,
        lambda_mel_hf=LAMBDA_MEL_HF,
        lambda_mel_tdiff=LAMBDA_MEL_TDIFF,
        lambda_mask_real=LAMBDA_MASK_REAL,
        lambda_mask_imag=LAMBDA_MASK_IMAG,
        lambda_blend_delta=LAMBDA_BLEND_DELTA,
        hf_start_frac=HF_START_FRAC,
        hf_end_gain=HF_END_GAIN,
        hf_ramp_power=HF_RAMP_POWER,
        stft_hf_start_frac=STFT_HF_START_FRAC,
        stft_hf_end_gain=STFT_HF_END_GAIN,
        stft_hf_ramp_power=STFT_HF_RAMP_POWER,
        mask_dilate=MASK_DILATE,
        tdiff_mask_dilate=TDIFF_MASK_DILATE,
        prior_radius=PRIOR_RADIUS,
        prior_boundary_gain=PRIOR_BOUNDARY_GAIN,
        prior_hf_power=PRIOR_HF_POWER,
    ),
    teachers=dict(
        mel_teacher_ckpt=str(MEL_TEACHER_CKPT) if MEL_TEACHER_CKPT is not None else None,
        mel_teacher_prefix=MEL_TEACHER_PREFIX,
        complex_teacher_ckpt=str(COMPLEX_TEACHER_CKPT) if COMPLEX_TEACHER_CKPT is not None else None,
        complex_teacher_prefix=COMPLEX_TEACHER_PREFIX,
    ),
)

(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print("CKPT_DIR:", CKPT_DIR)
print("LOG_CSV:", LOG_CSV)

## BigVGAN helpers 
BigVGAN is frozen. In this notebook it is used to generate the mel-route waveform:

\[
M_{\mathrm{mel\ teacher}} \rightarrow \text{BigVGAN} \rightarrow x_{\mathrm{mel}}
\]

Then \(x_{\mathrm{mel}}\) is converted to the blend STFT grid.

In [ ]:
# =========================
# BigVGAN setup
# =========================
BIGVGAN_REPO = Path("/content/BigVGAN")
if not BIGVGAN_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/NVIDIA/BigVGAN.git", str(BIGVGAN_REPO)], check=True)

if str(BIGVGAN_REPO) not in sys.path:
    sys.path.insert(0, str(BIGVGAN_REPO))

import bigvgan
from meldataset import mel_spectrogram as bigvgan_mel_spectrogram

bigvgan_G = None

def ensure_bigvgan_loaded():
    global bigvgan_G
    if bigvgan_G is None:
        print("Loading BigVGAN:", BIGVGAN_MODEL_ID)
        bigvgan_G = bigvgan.BigVGAN.from_pretrained(
            BIGVGAN_MODEL_ID,
            use_cuda_kernel=BIGVGAN_USE_CUDA_KERNEL,
        ).to(device).eval()
        for p in bigvgan_G.parameters():
            p.requires_grad_(False)
    return bigvgan_G

def wav_to_bigvgan_mel(wav_bt: torch.Tensor):
    if wav_bt.ndim == 1:
        wav_bt = wav_bt.unsqueeze(0)
    return bigvgan_mel_spectrogram(
        wav_bt,
        n_fft=N_FFT,
        num_mels=N_MELS,
        sampling_rate=SR,
        hop_size=HOP,
        win_size=WIN,
        fmin=FMIN,
        fmax=FMAX,
        center=CENTER_MEL,
    )

def mel_to_wave_bigvgan(mel_bft: torch.Tensor):
    Gv = ensure_bigvgan_loaded()
    y = Gv(mel_bft)
    if y.ndim == 3 and y.shape[1] == 1:
        y = y[:, 0]
    return y

def match_wav_length(wav_bt: torch.Tensor, target_len: int):
    if wav_bt.shape[-1] > target_len:
        return wav_bt[..., :target_len]
    if wav_bt.shape[-1] < target_len:
        return F.pad(wav_bt, (0, target_len - wav_bt.shape[-1]))
    return wav_bt

In [ ]:
# =========================
# Stage datasets locally for faster Colab training
# =========================
DRIVE_SPLIT_DIR = SPLIT_DIR
LOCAL_DATA_ROOT = Path("/content/audio_70speech_30music_v1_cache")
LOCAL_SPLIT_DIR = Path("/content/audio_70speech_30music_v1_splits_local")
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
assert SPLIT_DIR.exists(), f"Missing prepared split directory: {SPLIT_DIR}"

LISTS = ["speech_train.txt", "music_train.txt", "speech_val.txt", "music_val.txt", "speech_test.txt", "music_test.txt"]

def _read_lines(p: Path):
    return [line.strip().strip('"') for line in p.read_text().splitlines() if line.strip()]

def to_local_path(src: Path):
    src = Path(src)
    try:
        rel = src.relative_to(DRIVE_ROOT)
    except Exception:
        rel = Path(src.name)
    return LOCAL_DATA_ROOT / rel

all_files = []
for name in LISTS:
    p = DRIVE_SPLIT_DIR / name
    if p.exists():
        all_files.extend([Path(x) for x in _read_lines(p)])
all_files = sorted(set(all_files))
print("unique files referenced:", len(all_files))

copied = 0
missing = 0
for src in all_files:
    dst = to_local_path(src)
    if not src.exists():
        missing += 1
        continue
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied += 1

print("copied:", copied, "missing:", missing)

for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    if not src_list.exists():
        continue
    paths = [Path(x) for x in _read_lines(src_list)]
    local_paths = [str(to_local_path(p)) for p in paths if to_local_path(p).exists()]
    (LOCAL_SPLIT_DIR / name).write_text("\n".join(local_paths), encoding="utf-8")
    print(name, "->", len(local_paths), "local paths")

SPLIT_DIR = LOCAL_SPLIT_DIR
RUN_CONFIG["split_dir"] = str(SPLIT_DIR)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print("Using SPLIT_DIR:", SPLIT_DIR)

In [ ]:
# =========================
# Dataset / dataloaders
# =========================
def read_list(path: Path):
    return [Path(line.strip().strip('"')) for line in path.read_text().splitlines() if line.strip()]

def safe_load_wav_mono(path: Path, target_sr=SR):
    try:
        wav, sr = torchaudio.load(str(path))
    except Exception:
        arr, sr = sf.read(str(path), always_2d=True)
        wav = torch.from_numpy(arr.T).float()
    if wav.ndim == 2 and wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    elif wav.ndim == 1:
        wav = wav.unsqueeze(0)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    wav = wav[0].float()
    mx = wav.abs().max().clamp_min(1e-6)
    wav = (0.95 * wav / mx).clamp(-1.0, 1.0)
    return wav

class FileListDataset(Dataset):
    def __init__(self, list_path: Path, sr=SR, segment_s=SEG_S):
        self.paths = read_list(list_path)
        self.sr = sr
        self.seg_len = int(round(sr * segment_s))
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        wav = safe_load_wav_mono(self.paths[idx], self.sr)
        if wav.numel() < self.seg_len:
            wav = F.pad(wav, (0, self.seg_len - wav.numel()))
        max_start = max(0, wav.numel() - self.seg_len)
        start = random.randint(0, max_start) if max_start > 0 else 0
        return wav[start:start+self.seg_len]

speech_train = FileListDataset(SPLIT_DIR / "speech_train.txt")
music_train  = FileListDataset(SPLIT_DIR / "music_train.txt")

speech_loader = DataLoader(
    speech_train, batch_size=BATCH, shuffle=True, drop_last=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
music_loader = DataLoader(
    music_train, batch_size=BATCH, shuffle=True, drop_last=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

def infinite(loader):
    while True:
        for batch in loader:
            yield batch

speech_iter = infinite(speech_loader)
music_iter = infinite(music_loader)

def next_mixed_batch():
    if random.random() < P_SPEECH:
        b = next(speech_iter)
    else:
        b = next(music_iter)
    return b.to(device, non_blocking=True)

print("speech train:", len(speech_train), "music train:", len(music_train))

## Signal helpers and losses 
The important distinction here is:

- mel losses keep the result compatible with the good mel/vocoder route
- STFT/log-STFT losses make the blend care about the sharper time-frequency structure
- waveform MRSTFT checks the reconstructed waveform itself
- mask penalties keep the model from replacing the mel route everywhere

In [ ]:
# =========================
# Signal helpers / masks / losses
# =========================
def build_freq_weights(n_bins: int, start_frac: float, end_gain: float, power: float, device=None):
    idx = torch.linspace(0.0, 1.0, steps=n_bins, device=device)
    ramp = ((idx - start_frac).clamp_min(0.0) / max(1e-6, 1.0 - start_frac)) ** power
    w = 1.0 + (end_gain - 1.0) * ramp
    return w.view(1, n_bins, 1)

MEL_FREQ_WEIGHTS = build_freq_weights(N_MELS, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)
STFT_FREQ_WEIGHTS = build_freq_weights(BLEND_F_BINS, STFT_HF_START_FRAC, STFT_HF_END_GAIN, STFT_HF_RAMP_POWER, device=device)

def stft_cfg(wav_bt: torch.Tensor, *, n_fft: int, hop: int, win: int, center: bool):
    window = torch.hann_window(win, device=wav_bt.device)
    return torch.stft(
        wav_bt,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win,
        window=window,
        center=center,
        return_complex=True,
    )

def istft_cfg(X_bft: torch.Tensor, *, n_fft: int, hop: int, win: int, center: bool, length: int):
    window = torch.hann_window(win, device=X_bft.device)
    return torch.istft(
        X_bft,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win,
        window=window,
        center=center,
        length=length,
    )

def stft_complex_teacher(wav_bt: torch.Tensor):
    return stft_cfg(wav_bt, n_fft=CPLX_N_FFT, hop=CPLX_HOP, win=CPLX_WIN, center=CPLX_CENTER)

def istft_complex_teacher(X_bft: torch.Tensor, length: int):
    return istft_cfg(X_bft, n_fft=CPLX_N_FFT, hop=CPLX_HOP, win=CPLX_WIN, center=CPLX_CENTER, length=length)

def stft_blend(wav_bt: torch.Tensor):
    return stft_cfg(wav_bt, n_fft=BLEND_N_FFT, hop=BLEND_HOP, win=BLEND_WIN, center=BLEND_CENTER)

def istft_blend(X_bft: torch.Tensor, length: int):
    return istft_cfg(X_bft, n_fft=BLEND_N_FFT, hop=BLEND_HOP, win=BLEND_WIN, center=BLEND_CENTER, length=length)

def safe_logmag(X: torch.Tensor, eps=1e-5):
    return torch.log(X.abs().clamp_min(eps))

def dilate_mask_time(mask: torch.Tensor, radius: int):
    # mask shape: (B,1,1,T)
    if radius <= 0:
        return mask
    return F.max_pool2d(mask, kernel_size=(1, 2 * radius + 1), stride=1, padding=(0, radius))

def expand_mask(mask: torch.Tensor, n_bins: int, radius: int = 0):
    m = dilate_mask_time(mask, radius)
    return m[:, 0].expand(-1, n_bins, -1)

def match_frames_tensor(x: torch.Tensor, T: int):
    if x.shape[-1] > T:
        return x[..., :T]
    if x.shape[-1] < T:
        pad = T - x.shape[-1]
        return F.pad(x, (0, pad))
    return x

def match_mask_frames(mask: torch.Tensor, T: int):
    if mask.shape[-1] > T:
        return mask[..., :T]
    if mask.shape[-1] < T:
        return F.pad(mask, (0, T - mask.shape[-1]))
    return mask

def trim_complex_to_common(*xs):
    T = min(x.shape[-1] for x in xs)
    return [x[..., :T] for x in xs]

def trim_real_to_common(*xs):
    T = min(x.shape[-1] for x in xs)
    return [x[..., :T] for x in xs]

def masked_mean_abs(x: torch.Tensor, mask_ft: torch.Tensor, freq_weights=None, eps: float = 1e-8):
    z = x.abs()
    if freq_weights is not None:
        fw = freq_weights.to(z.device)
        z = z * fw
        denom = (mask_ft * fw).sum()
    else:
        denom = mask_ft.sum()
    return (z * mask_ft).sum() / (denom + eps)

def masked_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)
    return masked_mean_abs(pred - tgt, m, freq_weights=freq_weights)

def masked_tdiff_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    pred, tgt = trim_real_to_common(pred, tgt)
    dp = pred[..., 1:] - pred[..., :-1]
    dt = tgt[..., 1:] - tgt[..., :-1]
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)[..., 1:]
    m = match_frames_tensor(m, dp.shape[-1])
    return masked_mean_abs(dp - dt, m, freq_weights=freq_weights)

def masked_complex_l1(X_pred: torch.Tensor, X_tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    X_pred, X_tgt = trim_complex_to_common(X_pred, X_tgt)
    m = expand_mask(mask, X_pred.shape[1], radius=mask_dilate)
    m = match_frames_tensor(m, X_pred.shape[-1])
    err = (X_pred.real - X_tgt.real).abs() + (X_pred.imag - X_tgt.imag).abs()
    if freq_weights is not None:
        fw = freq_weights.to(err.device)
        denom = (m * fw).sum()
        return (err * m * fw).sum() / (denom + 1e-8)
    return (err * m).sum() / (m.sum() + 1e-8)

def masked_logmag_l1(X_pred: torch.Tensor, X_tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    X_pred, X_tgt = trim_complex_to_common(X_pred, X_tgt)
    lp = safe_logmag(X_pred)
    lt = safe_logmag(X_tgt)
    m = expand_mask(mask, X_pred.shape[1], radius=mask_dilate)
    m = match_frames_tensor(m, X_pred.shape[-1])
    return masked_mean_abs(lp - lt, m, freq_weights=freq_weights)

def masked_phase_cos_loss(X_pred: torch.Tensor, X_tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0, eps=1e-8):
    X_pred, X_tgt = trim_complex_to_common(X_pred, X_tgt)
    pn = X_pred / X_pred.abs().clamp_min(eps)
    tn = X_tgt / X_tgt.abs().clamp_min(eps)
    # 1 - cos(phase difference)
    err = 1.0 - (pn.real * tn.real + pn.imag * tn.imag)
    m = expand_mask(mask, X_pred.shape[1], radius=mask_dilate)
    m = match_frames_tensor(m, X_pred.shape[-1])
    if freq_weights is not None:
        fw = freq_weights.to(err.device)
        denom = (m * fw).sum()
        return (err * m * fw).sum() / (denom + 1e-8)
    return (err * m).sum() / (m.sum() + 1e-8)

def mrstft_loss(wav_hat: torch.Tensor, wav_real: torch.Tensor):
    wav_hat, wav_real = trim_real_to_common(wav_hat, wav_real)
    total = 0.0
    count = 0
    for n_fft, hop, win in zip(WAV_MRSTFT_FFTS, WAV_MRSTFT_HOPS, WAV_MRSTFT_WINS):
        window = torch.hann_window(win, device=wav_hat.device)
        Sh = torch.stft(wav_hat, n_fft=n_fft, hop_length=hop, win_length=win, window=window, center=True, return_complex=True)
        Sr = torch.stft(wav_real, n_fft=n_fft, hop_length=hop, win_length=win, window=window, center=True, return_complex=True)
        total = total + (safe_logmag(Sh) - safe_logmag(Sr)).abs().mean()
        count += 1
    return total / max(count, 1)

In [ ]:
# =========================
# Corruption / shared pair
# =========================
def linear_time_inpaint_mel(mel: torch.Tensor, k: int, offset: int):
    B, Freq, T = mel.shape
    step = k + 1
    mel_interp = mel.clone()
    mask = torch.zeros((B, 1, 1, T), device=mel.device, dtype=torch.float32)
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            mel_interp[:, :, t] = (1.0 - alpha) * mel[:, :, left] + alpha * mel[:, :, right]
            mask[:, :, :, t] = 1.0
    return mel_interp, mask

def linear_time_inpaint_complex(X: torch.Tensor, k: int, offset: int):
    B, Freq, T = X.shape
    step = k + 1
    Xr = X.real.clone()
    Xi = X.imag.clone()
    mask = torch.zeros((B, 1, 1, T), device=X.device, dtype=torch.float32)
    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            Xr[:, :, t] = (1.0 - alpha) * X.real[:, :, left] + alpha * X.real[:, :, right]
            Xi[:, :, t] = (1.0 - alpha) * X.imag[:, :, left] + alpha * X.imag[:, :, right]
            mask[:, :, :, t] = 1.0
    return torch.complex(Xr, Xi), mask

def make_shared_pair(wav_bt: torch.Tensor, k_choices=(1,2), randomize_offset=True):
    k = int(random.choice(list(k_choices)))
    step = k + 1
    offset = int(random.randrange(step)) if randomize_offset else 0

    mel_real = wav_to_bigvgan_mel(wav_bt)
    mel_interp, mask_mel = linear_time_inpaint_mel(mel_real, k=k, offset=offset)

    X_real_teacher = stft_complex_teacher(wav_bt)
    X_interp_teacher, mask_cplx = linear_time_inpaint_complex(X_real_teacher, k=k, offset=offset)

    return dict(
        wav=wav_bt,
        mel_real=mel_real,
        mel_interp=mel_interp,
        mask_mel=mask_mel,
        X_real_teacher=X_real_teacher,
        X_interp_teacher=X_interp_teacher,
        mask_cplx=mask_cplx,
        k=k,
        offset=offset,
    )

def make_stft_prior(mask_stft: torch.Tensor, n_bins: int, radius: int = PRIOR_RADIUS):
    # mask_stft shape (B,1,1,T)
    base = expand_mask(mask_stft, n_bins, radius=0)
    wide = expand_mask(mask_stft, n_bins, radius=radius)
    boundary = (wide - base).clamp_min(0.0)

    fw = STFT_FREQ_WEIGHTS.to(mask_stft.device).expand(base.shape[0], -1, base.shape[-1])
    hf_norm = ((fw - 1.0) / max(1e-6, STFT_HF_END_GAIN - 1.0)).clamp(0.0, 1.0)
    hf_shape = hf_norm ** PRIOR_HF_POWER

    region = (base + PRIOR_BOUNDARY_GAIN * boundary).clamp(0.0, 1.0)
    return (region * hf_shape).clamp(0.0, 1.0)

## Architecture 
The trainable model is a 2D U-Net on the blend STFT grid.

It does not predict the final STFT directly. It predicts the complex mask \(C(f,t)\) used to blend the two routes:

\[
X_{\mathrm{blend}}
=
X_{\mathrm{mel}}
+
P C (X_{\mathrm{stft}} - X_{\mathrm{mel}}).
\]

The real part of \(C\) controls how much of the STFT route is borrowed.  
The imaginary part allows phase-rotated corrections.

In [ ]:
# =========================
# Model definitions: teachers + complex-mask fusion model
# =========================
def _valid_groups(ch: int, requested: int):
    for g in [requested, 8, 4, 2, 1]:
        if ch % g == 0:
            return g
    return 1

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        self.net = nn.Sequential(
            ConvGNAct(in_ch, out_ch, groups=groups),
            ConvGNAct(out_ch, out_ch, groups=groups),
        )
    def forward(self, x):
        return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(2,2), groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            ConvGNAct(out_ch, out_ch, groups=groups),
        )
    def forward(self, x):
        return self.net(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8):
        super().__init__()
        self.fuse = DoubleConv(in_ch + skip_ch, out_ch, groups=groups)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.fuse(x)

class MelRefinerUNet2D(nn.Module):
    def __init__(self, n_mels=80, base=24, use_mask=True, groups=8):
        super().__init__()
        self.use_mask = use_mask
        c_in = 1 + (1 if use_mask else 0)

        self.enc1 = DoubleConv(c_in,       base,     groups=groups)
        self.enc2 = DownBlock(base,        base*2,   stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2,      base*4,   stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4,      base*4,   stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4,     base*4,   groups=groups)

        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)

        self.out_conv = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
        x = mel_interp.unsqueeze(1)
        if self.use_mask:
            mask2d = mask.unsqueeze(2).expand(-1, 1, mel_interp.shape[1], -1)
            x = torch.cat([x, mask2d], dim=1)

        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        u0 = u0 + s1
        delta = self.out_conv(u0).squeeze(1)
        return delta

class ComplexConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ComplexSTFTUNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=2, base=24, groups=8):
        super().__init__()
        self.enc1 = ComplexConvGNAct(in_ch, base, groups)
        self.enc2 = ComplexConvGNAct(base, base*2, groups)
        self.enc3 = ComplexConvGNAct(base*2, base*4, groups)
        self.bot  = ComplexConvGNAct(base*4, base*8, groups)
        self.dec3 = ComplexConvGNAct(base*8 + base*4, base*4, groups)
        self.dec2 = ComplexConvGNAct(base*4 + base*2, base*2, groups)
        self.dec1 = ComplexConvGNAct(base*2 + base, base, groups)
        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))

        y3 = F.interpolate(xb, size=x3.shape[-2:], mode="bilinear", align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))
        y2 = F.interpolate(y3, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))
        y1 = F.interpolate(y2, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))
        return self.out(y1)

def pack_complex_input(X_interp: torch.Tensor, mask: torch.Tensor):
    xr = X_interp.real.unsqueeze(1)
    xi = X_interp.imag.unsqueeze(1)
    xm = mask.expand(-1, 1, X_interp.shape[1], -1)
    return torch.cat([xr, xi, xm], dim=1)

class ComplexMaskFusionUNet2D(nn.Module):
    '''
    Input channels on the blend STFT grid:
      0 logmag(mel-route)
      1 logmag(stft-route)
      2 logmag(stft-route) - logmag(mel-route)
      3 cos(phase_stft - phase_mel)
      4 sin(phase_stft - phase_mel)
      5 prior
      6 mask

    Output:
      real mask parameter and imaginary mask parameter.
    '''
    def __init__(self, in_ch=7, base=24, groups=8):
        super().__init__()
        self.enc1 = DoubleConv(in_ch,       base,     groups=groups)
        self.enc2 = DownBlock(base,         base*2,   stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2,       base*4,   stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4,       base*4,   stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4,      base*4,   groups=groups)

        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)

        self.real_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        self.imag_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)

        nn.init.zeros_(self.real_head.weight)
        nn.init.constant_(self.real_head.bias, INIT_ALPHA_BIAS)
        nn.init.zeros_(self.imag_head.weight)
        nn.init.zeros_(self.imag_head.bias)

    def forward(self, feat_bctf):
        s1 = self.enc1(feat_bctf)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        u0 = u0 + s1

        real_raw = self.real_head(u0).squeeze(1)
        imag_raw = self.imag_head(u0).squeeze(1)

        alpha = MASK_REAL_MAX * torch.sigmoid(MASK_TEMP * real_raw)
        beta = MASK_IMAG_SCALE * torch.tanh(imag_raw)

        C = torch.complex(alpha, beta)
        return C, alpha, beta

G = ComplexMaskFusionUNet2D(in_ch=FUSION_IN_CH, base=G_BASE, groups=G_GROUPS).to(device)
print("Complex-mask fusion G params (M):", sum(p.numel() for p in G.parameters()) / 1e6)

In [ ]:
# =========================
# Teacher loading, with path resolution and sanity checks
# =========================
def extract_state_dict(ckpt, preferred_keys=("G", "model", "model_state_dict", "state_dict", "generator")):
    if isinstance(ckpt, dict):
        for k in preferred_keys:
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]
        # If this itself looks like a state_dict
        if all(torch.is_tensor(v) for v in ckpt.values()):
            return ckpt
    raise KeyError(f"Could not find model state dict. Available keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}")

def find_latest_ckpt(prefix: str):
    cands = sorted((DRIVE_ROOT / "checkpoints_runs").glob(f"{prefix}_*/latest.pt"))
    if cands:
        return cands[-1]
    # fallback: some older runs may not have latest.pt, so use largest step checkpoint in latest folder
    run_dirs = sorted((DRIVE_ROOT / "checkpoints_runs").glob(f"{prefix}_*"))
    for d in reversed(run_dirs):
        pts = sorted(d.glob("*.pt"))
        if pts:
            return pts[-1]
    return None

def cfg_get(cfg, keys, default=None):
    """Read direct or nested config keys from old/new run_config layouts."""
    for key in keys:
        if isinstance(key, tuple):
            cur = cfg
            ok = True
            for part in key:
                if isinstance(cur, dict) and part in cur:
                    cur = cur[part]
                else:
                    ok = False
                    break
            if ok:
                return cur
        else:
            if isinstance(cfg, dict) and key in cfg:
                return cfg[key]
    return default

def infer_base_from_state_dict(sd, default=24):
    # Works for these U-Net families: enc1.net.0.net.0.weight has shape (base, in_ch, 3, 3)
    for key in ["enc1.net.0.net.0.weight", "enc1.block.0.weight"]:
        if key in sd:
            return int(sd[key].shape[0])
    for k, v in sd.items():
        if k.endswith("weight") and getattr(v, "ndim", 0) == 4:
            return int(v.shape[0])
    return default

# ---- resolve teacher checkpoint paths ----
if MEL_TEACHER_CKPT is None:
    MEL_TEACHER_CKPT = None  # Example: Path("/content/drive/MyDrive/master/checkpoints_runs/<teacher_run_name>/latest.pt")
assert MEL_TEACHER_CKPT is not None and Path(MEL_TEACHER_CKPT).exists(), (
    f"Missing MEL_TEACHER_CKPT. Set MEL_TEACHER_CKPT manually, or check MEL_TEACHER_PREFIX={MEL_TEACHER_PREFIX!r}."
)

if COMPLEX_TEACHER_CKPT is None:
    COMPLEX_TEACHER_CKPT = None  # Example: Path("/content/drive/MyDrive/master/checkpoints_runs/<teacher_run_name>/latest.pt")
assert COMPLEX_TEACHER_CKPT is not None and Path(COMPLEX_TEACHER_CKPT).exists(), (
    f"Missing COMPLEX_TEACHER_CKPT. Set COMPLEX_TEACHER_CKPT manually, or check COMPLEX_TEACHER_PREFIX={COMPLEX_TEACHER_PREFIX!r}."
)

# ---- mel teacher ----
mel_ck = torch.load(str(MEL_TEACHER_CKPT), map_location="cpu")
mel_cfg = mel_ck.get("run_config", {}) if isinstance(mel_ck, dict) else {}
mel_sd = extract_state_dict(mel_ck)

mel_base = int(cfg_get(mel_cfg, ["g_base", "G_BASE", "G_base", "base", ("arch", "g_base")], infer_base_from_state_dict(mel_sd, 24)))
mel_groups = int(cfg_get(mel_cfg, ["g_groups", "G_GROUPS", "G_groups", "groups", ("arch", "g_groups")], G_GROUPS))
mel_use_mask = bool(cfg_get(mel_cfg, ["g_use_mask", "G_USE_MASK", "G_use_mask", "use_mask", ("arch", "g_use_mask")], True))
mel_n_mels = int(cfg_get(mel_cfg, ["n_mels", "N_MELS", ("mel", "n_mels")], N_MELS))

MelTeacher = MelRefinerUNet2D(n_mels=mel_n_mels, base=mel_base, use_mask=mel_use_mask, groups=mel_groups).to(device)
try:
    MelTeacher.load_state_dict(mel_sd, strict=True)
except RuntimeError as e:
    print("\nCould not load mel teacher strictly.")
    print("Checkpoint:", MEL_TEACHER_CKPT)
    print("Inferred base/groups/use_mask/n_mels:", mel_base, mel_groups, mel_use_mask, mel_n_mels)
    print("This usually means the checkpoint is not from the plain 2D mel U-Net family.")
    raise e
MelTeacher.eval()
for p in MelTeacher.parameters():
    p.requires_grad_(False)

# ---- complex STFT teacher ----
cplx_ck = torch.load(str(COMPLEX_TEACHER_CKPT), map_location="cpu")
cplx_cfg = cplx_ck.get("run_config", {}) if isinstance(cplx_ck, dict) else {}
cplx_sd = extract_state_dict(cplx_ck)

cplx_base = int(cfg_get(cplx_cfg, ["g_base", "G_BASE", "G_base", "base", ("arch", "g_base")], infer_base_from_state_dict(cplx_sd, 24)))
cplx_groups = int(cfg_get(cplx_cfg, ["g_groups", "G_GROUPS", "G_groups", "groups", ("arch", "g_groups")], G_GROUPS))
cplx_in = int(cfg_get(cplx_cfg, ["g_in_ch", "G_IN_CH", ("arch", "g_in_ch")], 3))
cplx_out = int(cfg_get(cplx_cfg, ["g_out_ch", "G_OUT_CH", ("arch", "g_out_ch")], 2))

ComplexTeacher = ComplexSTFTUNet2D(in_ch=cplx_in, out_ch=cplx_out, base=cplx_base, groups=cplx_groups).to(device)
try:
    ComplexTeacher.load_state_dict(cplx_sd, strict=True)
except RuntimeError as e:
    print("\nCould not load complex STFT teacher strictly.")
    print("Checkpoint:", COMPLEX_TEACHER_CKPT)
    print("Inferred base/groups/in/out:", cplx_base, cplx_groups, cplx_in, cplx_out)
    raise e
ComplexTeacher.eval()
for p in ComplexTeacher.parameters():
    p.requires_grad_(False)

RUN_CONFIG["teachers"]["mel_teacher_ckpt"] = str(MEL_TEACHER_CKPT)
RUN_CONFIG["teachers"]["complex_teacher_ckpt"] = str(COMPLEX_TEACHER_CKPT)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")

print("Loaded mel teacher    :", MEL_TEACHER_CKPT)
print("  base/groups/use_mask:", mel_base, mel_groups, mel_use_mask)
print("Loaded complex teacher:", COMPLEX_TEACHER_CKPT)
print("  base/groups/in/out  :", cplx_base, cplx_groups, cplx_in, cplx_out)


In [ ]:
# =========================
# Route construction and complex-mask blend
# =========================
@torch.no_grad()
def mel_teacher_route(mel_interp: torch.Tensor, mask_mel: torch.Tensor, target_len: int):
    # mel teacher output
    delta = MelTeacher(mel_interp, mask_mel[:, :, 0, :])
    mel_hat = mel_interp + delta * mask_mel[:, 0]
    wav_mel = mel_to_wave_bigvgan(mel_hat)
    wav_mel = match_wav_length(wav_mel, target_len)
    return mel_hat, wav_mel

@torch.no_grad()
def complex_teacher_route(X_interp_teacher: torch.Tensor, mask_cplx: torch.Tensor, target_len: int):
    x_in = pack_complex_input(X_interp_teacher, mask_cplx)
    delta = ComplexTeacher(x_in)
    dX = torch.complex(delta[:, 0], delta[:, 1])
    mask_ft = mask_cplx[:, 0].expand(-1, X_interp_teacher.shape[1], -1)
    X_hat_teacher = X_interp_teacher + dX * mask_ft
    wav_stft = istft_complex_teacher(X_hat_teacher, length=target_len)
    wav_stft = match_wav_length(wav_stft, target_len)
    return X_hat_teacher, wav_stft

def build_fusion_features(X_mel: torch.Tensor, X_stft: torch.Tensor, prior: torch.Tensor, mask_stft: torch.Tensor):
    X_mel, X_stft = trim_complex_to_common(X_mel, X_stft)
    T = X_mel.shape[-1]
    prior = match_frames_tensor(prior, T)
    mask_ft = expand_mask(match_mask_frames(mask_stft, T), X_mel.shape[1], radius=0)

    lm_mel = safe_logmag(X_mel)
    lm_stft = safe_logmag(X_stft)
    lm_diff = lm_stft - lm_mel

    phase_diff = torch.angle(X_stft) - torch.angle(X_mel)
    cos_d = torch.cos(phase_diff)
    sin_d = torch.sin(phase_diff)

    feat = torch.stack([lm_mel, lm_stft, lm_diff, cos_d, sin_d, prior, mask_ft], dim=1)
    # Simple per-channel clamp for numerical sanity.
    feat[:, 0:3] = feat[:, 0:3].clamp(-12.0, 4.0)
    assert feat.ndim == 4 and feat.shape[1] == FUSION_IN_CH, f"bad fusion feature shape: {tuple(feat.shape)}"
    assert prior.shape == X_mel.shape, f"bad prior shape {tuple(prior.shape)} vs X {tuple(X_mel.shape)}"
    assert mask_ft.shape == X_mel.shape, f"bad mask shape {tuple(mask_ft.shape)} vs X {tuple(X_mel.shape)}"
    return feat, prior, mask_ft

def blend_with_complex_mask(X_mel: torch.Tensor, X_stft: torch.Tensor, C: torch.Tensor, prior: torch.Tensor):
    X_mel, X_stft = trim_complex_to_common(X_mel, X_stft)
    T = X_mel.shape[-1]
    C = match_frames_tensor(C, T)
    prior = match_frames_tensor(prior, T)
    return X_mel + prior * C * (X_stft - X_mel)

In [ ]:
# =========================
# Optimizer / checkpoint helpers
# =========================
opt_g = torch.optim.AdamW(G.parameters(), lr=LR_G, betas=(0.9, 0.99), weight_decay=WEIGHT_DECAY)

def save_ckpt(step: int):
    obj = {
        "step": step,
        "mode": MODE,
        "run_config": RUN_CONFIG,
        "G": G.state_dict(),
        "opt_g": opt_g.state_dict(),
        "mel_teacher_ckpt": str(MEL_TEACHER_CKPT),
        "complex_teacher_ckpt": str(COMPLEX_TEACHER_CKPT),
        "stft_freq_weights": STFT_FREQ_WEIGHTS.detach().cpu(),
        "mel_freq_weights": MEL_FREQ_WEIGHTS.detach().cpu(),
    }
    p = CKPT_DIR / f"{MODE}_step{step:07d}.pt"
    torch.save(obj, p)
    torch.save(obj, CKPT_DIR / "latest.pt")
    print("saved:", p)

start_step = 0
if RESUME_PATH is None and AUTO_RESUME_SAME_FAMILY:
    cands = sorted((DRIVE_ROOT / "checkpoints_runs").glob(f"{RUN_NAME}_*/latest.pt"))
    if cands:
        RESUME_PATH = cands[-1]

if RESUME_PATH is not None:
    RESUME_PATH = Path(RESUME_PATH)
    if RESUME_PATH.exists():
        ck = torch.load(str(RESUME_PATH), map_location="cpu")
        G.load_state_dict(ck["G"], strict=True)
        if "opt_g" in ck:
            opt_g.load_state_dict(ck["opt_g"])
        start_step = int(ck.get("step", 0))
        for pg in opt_g.param_groups:
            pg["lr"] = LR_G * RESUME_LR_SCALE
        print("Resumed:", RESUME_PATH, "start_step:", start_step)
    else:
        print("RESUME_PATH not found, starting fresh:", RESUME_PATH)
else:
    print("Starting fresh.")

In [ ]:
# =========================
# Smoke test: verifies shapes + differentiability before long training
# =========================
def one_batch_complexmask_forward(wav: torch.Tensor):
    """Shared forward used by the smoke test. The real training loop below duplicates this logic for logging."""
    wav = wav.clamp(-1.0, 1.0)
    pair = make_shared_pair(wav, k_choices=K_CHOICES, randomize_offset=True)
    mel_real = pair["mel_real"]
    mel_interp = pair["mel_interp"]
    mask_mel = pair["mask_mel"]
    X_interp_teacher = pair["X_interp_teacher"]
    mask_cplx = pair["mask_cplx"]
    target_len = wav.shape[-1]

    with torch.no_grad():
        mel_teacher_hat, wav_mel = mel_teacher_route(mel_interp, mask_mel, target_len=target_len)
        _, wav_stft = complex_teacher_route(X_interp_teacher, mask_cplx, target_len=target_len)
        X_real = stft_blend(wav)
        X_mel = stft_blend(wav_mel)
        X_stft = stft_blend(wav_stft)
        X_real, X_mel, X_stft = trim_complex_to_common(X_real, X_mel, X_stft)
        T_blend = X_real.shape[-1]
        mask_blend = match_mask_frames(mask_cplx, T_blend)
        prior = make_stft_prior(mask_blend, X_real.shape[1], radius=PRIOR_RADIUS)
        prior = match_frames_tensor(prior, T_blend)

    feat, prior, _ = build_fusion_features(X_mel, X_stft, prior, mask_blend)
    C, alpha, beta = G(feat)
    X_blend = blend_with_complex_mask(X_mel, X_stft, C, prior)
    wav_blend = istft_blend(X_blend, length=target_len)

    loss = masked_logmag_l1(X_blend, X_real, mask_blend, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    return dict(
        loss=loss,
        wav=wav,
        wav_mel=wav_mel,
        wav_stft=wav_stft,
        wav_blend=wav_blend,
        X_real=X_real,
        X_mel=X_mel,
        X_stft=X_stft,
        X_blend=X_blend,
        feat=feat,
        C=C,
        alpha=alpha,
        beta=beta,
        prior=prior,
        mask_blend=mask_blend,
    )

if RUN_SMOKE_TEST_BEFORE_TRAINING:
    print("Running smoke test on one batch...")
    G.train()
    wav_smoke = next_mixed_batch()[:1]
    opt_g.zero_grad(set_to_none=True)
    out = one_batch_complexmask_forward(wav_smoke)
    out["loss"].backward()
    grad_norm = torch.sqrt(sum((p.grad.detach().float().norm() ** 2) for p in G.parameters() if p.grad is not None))
    opt_g.zero_grad(set_to_none=True)

    print("smoke loss:", float(out["loss"].detach().cpu()))
    print("grad norm:", float(grad_norm.detach().cpu()))
    print("feat shape:", tuple(out["feat"].shape), "expected channels:", FUSION_IN_CH)
    print("X shapes: real/mel/stft/blend", tuple(out["X_real"].shape), tuple(out["X_mel"].shape), tuple(out["X_stft"].shape), tuple(out["X_blend"].shape))
    print("alpha mean in prior:", float((out["alpha"].detach() * out["prior"]).sum() / (out["prior"].sum() + 1e-8)))
    print("beta abs mean in prior:", float((out["beta"].detach().abs() * out["prior"]).sum() / (out["prior"].sum() + 1e-8)))
    assert torch.isfinite(out["loss"]).item(), "Smoke-test loss is not finite."
    assert grad_norm > 0, "No gradient reached the complex-mask fusion model."
    print("Smoke test passed.")


## Training loop 
Each training step:

1. sample waveform \(x\)
2. create matching mel and complex-STFT inpainting corruptions
3. frozen mel teacher gives mel-route output, then BigVGAN gives \(x_{\mathrm{mel}}\)
4. frozen STFT teacher gives STFT-route output, then ISTFT gives \(x_{\mathrm{stft}}\)
5. compute \(X_{\mathrm{mel}}\), \(X_{\mathrm{stft}}\), and \(X_{\mathrm{real}}\) on the same STFT grid
6. train the fusion U-Net to predict \(C(f,t)\)
7. reconstruct \(x_{\mathrm{blend}}\) with ISTFT
8. optimize STFT, waveform, and mel losses

In [ ]:
# =========================
# Training
# =========================
G.train()
t0 = time.time()

if not LOG_CSV.exists():
    LOG_CSV.write_text(
        "step,loss_total,loss_logstft_hf,loss_complex,loss_phase,loss_wav_mrstft,"
        "loss_mel_recon,loss_mel_hf,loss_mel_tdiff,loss_mask_real,loss_mask_imag,loss_blend_delta,"
        "mel_logstft_hf,stft_logstft_hf,blend_logstft_hf,mel_mel_hf,stft_mel_hf,blend_mel_hf,"
        "alpha_mean,beta_abs_mean,blend_delta_mag,k,offset,elapsed_min\\n",
        encoding="utf-8"
    )

for step in range(start_step + 1, start_step + STEPS + 1):
    wav = next_mixed_batch()
    wav = wav.clamp(-1.0, 1.0)

    pair = make_shared_pair(wav, k_choices=K_CHOICES, randomize_offset=True)
    mel_real = pair["mel_real"]
    mel_interp = pair["mel_interp"]
    mask_mel = pair["mask_mel"]
    X_interp_teacher = pair["X_interp_teacher"]
    mask_cplx = pair["mask_cplx"]
    k = pair["k"]
    offset = pair["offset"]

    target_len = wav.shape[-1]

    # Frozen teacher routes
    with torch.no_grad():
        mel_teacher_hat, wav_mel = mel_teacher_route(mel_interp, mask_mel, target_len=target_len)
        X_teacher_hat, wav_stft = complex_teacher_route(X_interp_teacher, mask_cplx, target_len=target_len)

        X_real = stft_blend(wav)
        X_mel = stft_blend(wav_mel)
        X_stft = stft_blend(wav_stft)

        X_real, X_mel, X_stft = trim_complex_to_common(X_real, X_mel, X_stft)
        T_blend = X_real.shape[-1]
        mask_blend = match_mask_frames(mask_cplx, T_blend)
        prior = make_stft_prior(mask_blend, X_real.shape[1], radius=PRIOR_RADIUS)
        prior = match_frames_tensor(prior, T_blend)

    # Trainable complex-mask predictor
    feat, prior, mask_ft0 = build_fusion_features(X_mel, X_stft, prior, mask_blend)
    C, alpha, beta = G(feat)

    X_blend = blend_with_complex_mask(X_mel, X_stft, C, prior)
    wav_blend = istft_blend(X_blend, length=target_len)

    # Main STFT/waveform losses
    loss_logstft_hf = masked_logmag_l1(X_blend, X_real, mask_blend, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_complex = masked_complex_l1(X_blend, X_real, mask_blend, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_phase = masked_phase_cos_loss(X_blend, X_real, mask_blend, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_wav_mrstft = mrstft_loss(wav_blend, wav)

    # Mel consistency losses
    mel_blend = wav_to_bigvgan_mel(wav_blend.clamp(-1.2, 1.2))
    mel_stft = wav_to_bigvgan_mel(wav_stft.clamp(-1.2, 1.2))

    mel_real_t, mel_blend_t, mel_teacher_t, mel_stft_t = trim_real_to_common(mel_real, mel_blend, mel_teacher_hat, mel_stft)
    mask_mel_t = match_mask_frames(mask_mel, mel_real_t.shape[-1])

    loss_mel_recon = masked_l1(mel_blend_t, mel_real_t, mask_mel_t, freq_weights=None, mask_dilate=MASK_DILATE)
    loss_mel_hf = masked_l1(mel_blend_t, mel_real_t, mask_mel_t, freq_weights=MEL_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_mel_tdiff = masked_tdiff_l1(mel_blend_t, mel_real_t, mask_mel_t, freq_weights=MEL_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)

    # Mask regularization: keep the blend local and avoid unnecessary phase rotation
    prior_w = prior.detach()
    loss_mask_real = (alpha.abs() * prior_w).sum() / (prior_w.sum() + 1e-8)
    loss_mask_imag = (beta.abs() * prior_w).sum() / (prior_w.sum() + 1e-8)
    loss_blend_delta = ((X_blend - X_mel).abs() * prior_w).sum() / (prior_w.sum() + 1e-8)

    loss_total = (
        LAMBDA_LOGSTFT_HF * loss_logstft_hf
        + LAMBDA_COMPLEX * loss_complex
        + LAMBDA_PHASE * loss_phase
        + LAMBDA_WAV_MRSTFT * loss_wav_mrstft
        + LAMBDA_MEL_RECON * loss_mel_recon
        + LAMBDA_MEL_HF * loss_mel_hf
        + LAMBDA_MEL_TDIFF * loss_mel_tdiff
        + LAMBDA_MASK_REAL * loss_mask_real
        + LAMBDA_MASK_IMAG * loss_mask_imag
        + LAMBDA_BLEND_DELTA * loss_blend_delta
    )

    opt_g.zero_grad(set_to_none=True)
    loss_total.backward()
    torch.nn.utils.clip_grad_norm_(G.parameters(), 5.0)
    opt_g.step()

    if step % LOG_EVERY == 0 or step == 1:
        with torch.no_grad():
            mel_logstft_hf = masked_logmag_l1(X_mel, X_real, mask_blend, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
            stft_logstft_hf = masked_logmag_l1(X_stft, X_real, mask_blend, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
            blend_logstft_hf = loss_logstft_hf.detach()

            mel_mel_hf = masked_l1(mel_teacher_t, mel_real_t, mask_mel_t, freq_weights=MEL_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
            stft_mel_hf = masked_l1(mel_stft_t, mel_real_t, mask_mel_t, freq_weights=MEL_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
            blend_mel_hf = loss_mel_hf.detach()

            alpha_mean = (alpha.detach() * prior_w).sum() / (prior_w.sum() + 1e-8)
            beta_abs_mean = (beta.detach().abs() * prior_w).sum() / (prior_w.sum() + 1e-8)
            blend_delta_mag = loss_blend_delta.detach()

        elapsed_min = (time.time() - t0) / 60.0
        line = (
            f"{step},{loss_total.item():.8f},{loss_logstft_hf.item():.8f},{loss_complex.item():.8f},"
            f"{loss_phase.item():.8f},{loss_wav_mrstft.item():.8f},{loss_mel_recon.item():.8f},"
            f"{loss_mel_hf.item():.8f},{loss_mel_tdiff.item():.8f},{loss_mask_real.item():.8f},"
            f"{loss_mask_imag.item():.8f},{loss_blend_delta.item():.8f},{mel_logstft_hf.item():.8f},"
            f"{stft_logstft_hf.item():.8f},{blend_logstft_hf.item():.8f},{mel_mel_hf.item():.8f},"
            f"{stft_mel_hf.item():.8f},{blend_mel_hf.item():.8f},{alpha_mean.item():.8f},"
            f"{beta_abs_mean.item():.8f},{blend_delta_mag.item():.8f},{k},{offset},{elapsed_min:.3f}\\n"
        )
        with LOG_CSV.open("a", encoding="utf-8") as f:
            f.write(line)

        print(
            f"step {step:7d} | loss {loss_total.item():.4f} | "
            f"logHF mel/stft/blend {mel_logstft_hf.item():.4f}/{stft_logstft_hf.item():.4f}/{blend_logstft_hf.item():.4f} | "
            f"melHF mel/stft/blend {mel_mel_hf.item():.4f}/{stft_mel_hf.item():.4f}/{blend_mel_hf.item():.4f} | "
            f"alpha {alpha_mean.item():.3f} beta {beta_abs_mean.item():.3f} | k {k} off {offset}"
        )

    if step % SAVE_EVERY == 0 or step == start_step + STEPS:
        save_ckpt(step)

print("done. CKPT_DIR:", CKPT_DIR)

In [ ]:
# =========================
# Training curves
# =========================
if LOG_CSV.exists():
    df = pd.read_csv(LOG_CSV)
    display(df.tail())

    metrics = [
        "loss_total",
        "mel_logstft_hf", "stft_logstft_hf", "blend_logstft_hf",
        "mel_mel_hf", "stft_mel_hf", "blend_mel_hf",
        "alpha_mean", "beta_abs_mean", "blend_delta_mag",
        "loss_mel_tdiff", "loss_wav_mrstft",
    ]
    metrics = [m for m in metrics if m in df.columns]

    ncols = 3
    nrows = math.ceil(len(metrics) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.2*nrows), squeeze=False)
    for ax, m in zip(axes.ravel(), metrics):
        ax.plot(df["step"], df[m])
        ax.set_title(m)
        ax.set_xlabel("step")
        ax.grid(True, alpha=0.3)
    for ax in axes.ravel()[len(metrics):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No log yet:", LOG_CSV)

In [ ]:
# =========================
# Exact listening sanity check
# =========================
EXACT_LISTEN_SPLIT_NAME = "speech_test.txt"
EXACT_LISTEN_SPLIT_INDEX = 4
EXACT_LISTEN_K = 1
EXACT_LISTEN_OFFSET = 0
EXACT_LISTEN_SEG_S = 1.5

@torch.no_grad()
def eval_one_file(path: Path, k=1, offset=0, seg_s=1.5):
    G.eval()
    wav = safe_load_wav_mono(path, SR)
    seg_len = int(round(seg_s * SR))
    if wav.numel() < seg_len:
        wav = F.pad(wav, (0, seg_len - wav.numel()))
    wav = wav[:seg_len].unsqueeze(0).to(device)
    wav = wav.clamp(-1.0, 1.0)
    target_len = wav.shape[-1]

    mel_real = wav_to_bigvgan_mel(wav)
    mel_interp, mask_mel = linear_time_inpaint_mel(mel_real, k=k, offset=offset)

    X_teacher_real = stft_complex_teacher(wav)
    X_interp_teacher, mask_cplx = linear_time_inpaint_complex(X_teacher_real, k=k, offset=offset)

    # References and teacher routes
    wav_real_voc = match_wav_length(mel_to_wave_bigvgan(mel_real), target_len).clamp(-1.0, 1.0)
    wav_interp = match_wav_length(mel_to_wave_bigvgan(mel_interp), target_len).clamp(-1.0, 1.0)
    mel_teacher_hat, wav_mel = mel_teacher_route(mel_interp, mask_mel, target_len)
    _, wav_stft = complex_teacher_route(X_interp_teacher, mask_cplx, target_len)

    X_real = stft_blend(wav)
    X_mel = stft_blend(wav_mel)
    X_stft = stft_blend(wav_stft)
    X_real, X_mel, X_stft = trim_complex_to_common(X_real, X_mel, X_stft)

    mask_blend = match_mask_frames(mask_cplx, X_real.shape[-1])
    prior = make_stft_prior(mask_blend, X_real.shape[1], radius=PRIOR_RADIUS)
    feat, prior, _ = build_fusion_features(X_mel, X_stft, prior, mask_blend)
    C, alpha, beta = G(feat)

    X_blend = blend_with_complex_mask(X_mel, X_stft, C, prior)
    wav_blend = istft_blend(X_blend, length=target_len).clamp(-1.0, 1.0)

    mel_interp_t = match_frames_tensor(mel_interp, mel_real.shape[-1])
    mel_blend = wav_to_bigvgan_mel(wav_blend)
    mel_stft = wav_to_bigvgan_mel(wav_stft.clamp(-1.0, 1.0))
    mel_real_t, mel_interp_t, mel_teacher_t, mel_stft_t, mel_blend_t = trim_real_to_common(
        mel_real, mel_interp_t, mel_teacher_hat, mel_stft, mel_blend
    )
    mask_mel_t = match_mask_frames(mask_mel, mel_real_t.shape[-1])

    rows = []
    for name, wav_out, mel_out, X_out in [
        ("interpolated mel -> BigVGAN", wav_interp, mel_interp_t, stft_blend(wav_interp)[..., :X_real.shape[-1]]),
        ("mel route (mel teacher -> BigVGAN)", wav_mel, mel_teacher_t, X_mel),
        ("stft route (complex teacher -> ISTFT)", wav_stft, mel_stft_t, X_stft),
        ("complex-mask blend", wav_blend, mel_blend_t, X_blend),
    ]:
        X_out, X_real_now = trim_complex_to_common(X_out, X_real)
        rows.append(dict(
            model=name,
            mel_hf=float(masked_l1(mel_out, mel_real_t, mask_mel_t, freq_weights=MEL_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()),
            mel_tdiff=float(masked_tdiff_l1(mel_out, mel_real_t, mask_mel_t, freq_weights=MEL_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE).item()),
            stft_loghf=float(masked_logmag_l1(X_out, X_real_now, mask_blend, freq_weights=STFT_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()),
        ))

    display(Markdown(f"### Exact listening case\n`{path}`\n\nk={k}, offset={offset}, segment={seg_s}s"))
    display(Markdown("#### Original waveform"))
    display(Audio(wav[0].detach().cpu().numpy(), rate=SR))
    display(Markdown("#### Original mel -> BigVGAN"))
    display(Audio(wav_real_voc[0].detach().cpu().numpy(), rate=SR))
    display(Markdown("#### Interpolated mel -> BigVGAN"))
    display(Audio(wav_interp[0].detach().cpu().numpy(), rate=SR))
    display(Markdown("#### Mel route: mel teacher -> BigVGAN"))
    display(Audio(wav_mel[0].detach().cpu().numpy(), rate=SR))
    display(Markdown("#### STFT route: complex teacher -> ISTFT"))
    display(Audio(wav_stft[0].detach().cpu().numpy(), rate=SR))
    display(Markdown("#### Complex-mask blend"))
    display(Audio(wav_blend[0].detach().cpu().numpy(), rate=SR))

    display(pd.DataFrame(rows))
    print("alpha mean in prior:", float((alpha * prior).sum() / (prior.sum() + 1e-8)))
    print("beta abs mean in prior:", float((beta.abs() * prior).sum() / (prior.sum() + 1e-8)))
    print("mel teacher ckpt:", MEL_TEACHER_CKPT)
    print("complex teacher ckpt:", COMPLEX_TEACHER_CKPT)

    G.train()

split_path = SPLIT_DIR / EXACT_LISTEN_SPLIT_NAME
listen_files = read_list(split_path)
listen_path = listen_files[EXACT_LISTEN_SPLIT_INDEX]
eval_one_file(listen_path, k=EXACT_LISTEN_K, offset=EXACT_LISTEN_OFFSET, seg_s=EXACT_LISTEN_SEG_S)
